# 02 — Silver: Yellow Taxi

Reads Yellow Taxi records from Bronze (`tlc_bronze.yellow_raw`), applies data
quality rules via **flag-based annotation** (no silent drops), enriches with
zone metadata, builds the normalized Silver schema, and writes:
- **Valid records** → `tlc_silver.trips_yellow` (with `quality_flags` preserved)
- **Invalid records** → `tlc_audit.quarantine` (with `_rejection_reason`)

Silver `_meta` carries `bronze_run_id` + `source_file` for full lineage tracing.

## Imports & Configuration

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from src.transformations.tlc_rules import (
    YELLOW_GREEN_RULES, apply_quality_flags, route_quarantine, print_rejection_summary
)
from src.transformations.enrichment import load_zone_lookup, enrich_trip_locations
from src.transformations.schema import build_yellow_silver
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
from core.storage.mongo_client import get_audit_db
import pyspark.sql.functions as F
import datetime

YEARS_TO_PROCESS = [2023, 2024, 2025]
print("Imports OK.")

Imports OK.


## Start Spark & Audit Control

In [2]:
spark = get_spark(app_name="TLC_Silver_Yellow")

control = ControlManager("silver_yellow")
execution = control.start({"years": YEARS_TO_PROCESS, "vehicle_type": "yellow"})
run_id = execution.execution_id
print(f"Execution ID (Silver run_id): {run_id}")

2026-07-19 03:55:18 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
2026-07-19 03:55:18 | INFO     | tlc.audit.control_manager | [AUDIT] Execution started | pipeline=silver_yellow id=d8d0254a
Execution ID (Silver run_id): d8d0254a


## Idempotency Purge

Ensures this notebook can be re-run safely without duplicating data by clearing previous artifacts for this pipeline.

In [3]:
from core.storage.mongo_client import get_audit_db, get_silver_db
from core.config.settings import settings

client = get_silver_db()
# 1. Purge Silver collection for yellow
get_silver_db()["trips_yellow"].delete_many({})

# 2. Purge Quarantine records generated by this pipeline
get_audit_db()["quarantine"].delete_many({"pipeline": "silver_yellow"})

print("Idempotency purge complete for yellow. Safe to run.")


2026-07-19 03:55:18 | INFO     | tlc.storage.mongo_client | [MONGO] Connected to mongodb:27017
Idempotency purge complete for yellow. Safe to run.


## Read from Bronze

In [4]:
df_bronze = read_mongo(spark, settings.MONGO_DB_BRONZE, "yellow_raw")

# Filter to requested years (safe: static batch DataFrame)
if "_meta" in df_bronze.columns:
    df_bronze = df_bronze.filter(F.year("tpep_pickup_datetime").isin(YEARS_TO_PROCESS))

# records_in will be evaluated after caching df_flagged


2026-07-19 03:55:24 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_bronze.yellow_raw


## Apply Data Quality Flags

> **No records are dropped here.**  
> Each record receives a `quality_flags` struct (one boolean per rule) and an
> `is_valid` column. Invalid records are *routed* to quarantine below.

In [5]:
# Step 1 — annotate every record with quality flags (no filtering)
df_flagged = apply_quality_flags(df_bronze, YELLOW_GREEN_RULES)

# Step 2 — split into valid / rejected partitions
records_in = df_flagged.count()
print(f"Records read from Bronze: {records_in:,}")

valid_df, rejected_df = route_quarantine(df_flagged)

records_valid    = valid_df.count()
records_rejected = rejected_df.count()

print(f"Valid records   : {records_valid:,}")
print(f"Rejected records: {records_rejected:,}")
print(f"Quarantine rate : {records_rejected / records_in * 100:.2f}%" if records_in else "N/A")

print_rejection_summary(rejected_df)

control.log_quality_check(
    check_name="yellow_quality_flags",
    dataset=f"yellow_raw_years_{YEARS_TO_PROCESS}",
    records_checked=records_in,
    records_failed=records_rejected,
)

2026-07-19 03:55:24 | INFO     | tlc.transformations.tlc_rules | [RULES] Quality flags applied | rules=['valid_distance', 'valid_fare', 'valid_total', 'valid_pickup_zone', 'valid_dropoff_zone', 'valid_time_order', 'valid_passengers', 'valid_vendor', 'valid_ratecode', 'valid_payment_type', 'valid_pickup_year_yellow']
Records read from Bronze: 128,202,401
Valid records   : 106,330,101
Rejected records: 21,872,300
Quarantine rate : 17.06%
2026-07-19 04:01:26 | INFO     | tlc.transformations.tlc_rules | [RULES] Total records rejected: 21,872,300
2026-07-19 04:03:17 | INFO     | tlc.transformations.tlc_rules |          ↳ 'Payment type must be in [1, 2, 3, 4, 5, 6].': 17,012,482
2026-07-19 04:03:17 | INFO     | tlc.transformations.tlc_rules |          ↳ 'Total amount must not be negative.': 1,789,376
2026-07-19 04:03:17 | INFO     | tlc.transformations.tlc_rules |          ↳ 'Trip distance must be greater than zero.': 1,262,884
2026-07-19 04:03:17 | INFO     | tlc.transformations.tlc_rules |

QualityCheckResult(check_id='d8d0254a_yellow_quality_flags', check_name='yellow_quality_flags', dataset='yellow_raw_years_[2023, 2024, 2025]', status=<QualityStatus.FAILED: 'FAILED'>, records_checked=128202401, records_passed=106330101, records_failed=21872300, failure_rate=0.1706075691983335, details={})

## Route Rejected Records to Quarantine

Invalid records are stored in `tlc_audit.quarantine` with:
- `_rejection_reason` — human-readable failing rule
- `quality_flags` — full per-rule boolean breakdown
- `bronze_run_id` (via `_meta.run_id`) — Bronze lineage
- `pipeline` and `silver_run_id` — Silver lineage

In [6]:
if records_rejected > 0:
    # OPTIMIZATION: Distributed PySpark write instead of driver collect()
    seen_cols = set()
    raw_cols = []
    for c in rejected_df.columns:
        if c not in ("_rejection_reason", "quality_flags", "_meta") and c.lower() not in seen_cols:
            raw_cols.append(c)
            seen_cols.add(c.lower())
    
    quarantine_df_mapped = rejected_df.select(
        F.current_timestamp().alias("quarantined_at"),
        F.lit(run_id).alias("silver_run_id"),
        F.col("_meta.run_id").alias("bronze_run_id"),
        F.col("_meta.source_file").alias("source_file"),
        F.lit("silver_yellow").alias("pipeline"),
        F.col("_rejection_reason").alias("reason"),
        F.col("quality_flags").alias("quality_flags"),
        F.struct(*[F.col(c) for c in raw_cols]).alias("raw_record")
    )
    
    write_mongo(quarantine_df_mapped, settings.MONGO_DB_AUDIT, "quarantine", mode="append")
    print(f"Quarantined {records_rejected:,} records into tlc_audit.quarantine (Distributed)")
else:
    print("No records quarantined.")

2026-07-19 04:08:36 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_audit.quarantine (mode=append)
Quarantined 21,872,300 records into tlc_audit.quarantine (Distributed)


## Enrich with Zone Metadata

In [7]:
zone_df = load_zone_lookup(spark)
valid_df = enrich_trip_locations(valid_df, zone_df)
print("Zone enrichment complete.")

2026-07-19 04:08:40 | INFO     | tlc.transformations.enrichment | [ENRICH] Zone lookup loaded: 265 zones from /home/jovyan/work/data/raw/lookup/taxi_zone_lookup.csv
2026-07-19 04:08:40 | INFO     | tlc.transformations.enrichment | [ENRICH] Location IDs enriched with Borough and Zone names.
Zone enrichment complete.


## Build Silver Schema & Write to MongoDB

Silver documents include:
- `quality_flags` — the per-rule audit trail
- `_meta.bronze_run_id` — traces back to the Bronze ingestion run
- `_meta.source_file` — original raw Parquet filename

In [8]:
silver_df = build_yellow_silver(valid_df, run_id=run_id)
n_written = write_mongo(silver_df, settings.MONGO_DB_SILVER, "trips_yellow", mode="append", num_rows=records_valid)
print(f"Rows written to tlc_silver.trips_yellow: {n_written:,}")
print(f"Lineage: Bronze → Silver run_id={run_id}")

2026-07-19 04:08:40 | INFO     | tlc.transformations.schema | [SCHEMA] Yellow Silver schema applied (with bronze lineage + quality_flags).
2026-07-19 04:27:05 | INFO     | tlc.spark_utils | [SPARK] Wrote 106,330,101 rows → MongoDB tlc_silver.trips_yellow (mode=append)
Rows written to tlc_silver.trips_yellow: 106,330,101
Lineage: Bronze → Silver run_id=d8d0254a


## Volumetric Traceability Check

Validates: **Bronze == Silver + Quarantine** (zero data lost in transit)

In [9]:
print(f"╔══════════════════════════════════════════╗")
print(f"║  Volumetric Traceability Report          ║")
print(f"╠══════════════════════════════════════════╣")
print(f"║  Bronze records in  : {records_in:>16,}  ║")
print(f"║  Silver records out : {n_written:>16,}  ║")
print(f"║  Quarantine records : {records_rejected:>16,}  ║")
print(f"║  Delta (must be 0)  : {records_in - n_written - records_rejected:>16,}  ║")
print(f"╚══════════════════════════════════════════╝")
assert records_in == n_written + records_rejected, "DATA LOSS DETECTED — investigate immediately!"

╔══════════════════════════════════════════╗
║  Volumetric Traceability Report          ║
╠══════════════════════════════════════════╣
║  Bronze records in  :      128,202,401  ║
║  Silver records out :      106,330,101  ║
║  Quarantine records :       21,872,300  ║
║  Delta (must be 0)  :                0  ║
╚══════════════════════════════════════════╝


In [10]:
control.end(
    ExecutionStatus.SUCCESS,
    {
        "records_read_from_bronze": records_in,
        "records_written_to_silver": n_written,
        "records_quarantined": records_rejected,
        "quarantine_rate_pct": round(records_rejected / records_in * 100, 4) if records_in else 0,
    },
)

2026-07-19 04:27:06 | INFO     | tlc.audit.control_manager | [AUDIT] Execution finished | id=d8d0254a status=SUCCESS duration=1907.6s
2026-07-19 04:27:06 | INFO     | tlc.audit.control_manager | [AUDIT] Report saved → /home/jovyan/work/data/audit/executions/20260719_042706_d8d0254a_silver_yellow.json
2026-07-19 04:27:07 | INFO     | tlc.audit.control_manager | [AUDIT] Report inserted into MongoDB tlc_audit.pipeline_runs (id=d8d0254a)


ExecutionRecord(pipeline_name='silver_yellow', input_parameters={'years': [2023, 2024, 2025], 'vehicle_type': 'yellow'}, execution_id='d8d0254a', start_time=datetime.datetime(2026, 7, 19, 3, 55, 18, 620523), end_time=datetime.datetime(2026, 7, 19, 4, 27, 6, 260721), status=<ExecutionStatus.SUCCESS: 'SUCCESS'>, output_summary={'records_read_from_bronze': 128202401, 'records_written_to_silver': 106330101, 'records_quarantined': 21872300, 'quarantine_rate_pct': 17.0608}, quality_checks=[QualityCheckResult(check_id='d8d0254a_yellow_quality_flags', check_name='yellow_quality_flags', dataset='yellow_raw_years_[2023, 2024, 2025]', status=<QualityStatus.FAILED: 'FAILED'>, records_checked=128202401, records_passed=106330101, records_failed=21872300, failure_rate=0.1706075691983335, details={})], errors=[])

## Audit Report

In [11]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))

{
  "execution_id": "d8d0254a",
  "pipeline_name": "silver_yellow",
  "status": "SUCCESS",
  "start_time": "2026-07-19T03:55:18.620523",
  "end_time": "2026-07-19T04:27:06.260721",
  "duration_seconds": 1907.64,
  "input_parameters": {
    "years": [
      2023,
      2024,
      2025
    ],
    "vehicle_type": "yellow"
  },
  "output_summary": {
    "records_read_from_bronze": 128202401,
    "records_written_to_silver": 106330101,
    "records_quarantined": 21872300,
    "quarantine_rate_pct": 17.0608
  },
  "quality_checks": [
    {
      "check_id": "d8d0254a_yellow_quality_flags",
      "check_name": "yellow_quality_flags",
      "dataset": "yellow_raw_years_[2023, 2024, 2025]",
      "status": "FAILED",
      "records_checked": 128202401,
      "records_passed": 106330101,
      "records_failed": 21872300,
      "failure_rate": 0.170608,
      "details": {}
    }
  ],
  "quality_passed": 0,
  "errors": []
}
